# Lab 2 — Checkpoint & resume (EP08)

From the theory video: task state has **progress**, **checkpoints** and **rollback** (§18.6.2),
and the textbook warns to design state **like a database schema**: a clear structure, with a version

**What you'll learn**
1. Write the task's state as a dataclass with a structure and a version number
2. A six-step task (one model call per step) where the network drops at step 4
3. No checkpoint: start over · with a checkpoint: resume at step 4 — and count the model calls
4. Save the file safely (`os.replace`), and roll back the last k steps

The papers and the network drop are **simulated data for teaching**. The model's answers and the call counts come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.6 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | the same key as Lab 1 (this lab costs under 1 US cent) |
| Packages | `openai`, `python-dotenv` |

How to add the key, and why we chose this model and provider, are in **Lab 1**

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
import json, os
from dataclasses import dataclass, field, asdict
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["makora"], "allow_fallbacks": False}}

USAGE = {"calls": 0, "in": 0, "out": 0}

def chat(messages, **kw):
    reply = client.chat.completions.create(model=MODEL, messages=messages, extra_body=EXTRA, **kw)
    USAGE["calls"] += 1
    USAGE["in"] += reply.usage.prompt_tokens
    USAGE["out"] += reply.usage.completion_tokens
    return reply.choices[0].message

In [ ]:
PAPERS = {
    "P1": "Informer (2021): ProbSparse self-attention makes long-sequence time-series forecasting faster.",
    "P2": "PatchTST (2023): splits each series into patches of 16 steps; each channel is handled on its own.",
    "P3": "TimesFM (2024): a decoder-only foundation model that forecasts new series zero-shot.",
}

## 1. The task's state: a structure, with a version

| Field | The textbook's layer of state |
|---|---|
| `task`, `steps` | the plan (agent state: *current plan*) |
| `results` | progress — which steps are done, and what they produced (task state: *progress*) |
| `version` | the version of the structure ("state as a first-class citizen") |

In [ ]:
STATE_VERSION = 1

@dataclass
class TaskState:
    task: str
    steps: list                                   # the plan: [name, prompt template]
    results: dict = field(default_factory=dict)   # progress: step name → output
    version: int = STATE_VERSION

In [ ]:
STEPS = [
    ["note_P1", "In one sentence, what is the key idea of this paper? {P1}"],
    ["note_P2", "In one sentence, what is the key idea of this paper? {P2}"],
    ["note_P3", "In one sentence, what is the key idea of this paper? {P3}"],
    ["compare", "Compare three papers in 3 short bullet points, using these notes:\n{note_P1}\n{note_P2}\n{note_P3}"],
    ["title",   "Write one title for a 2-page report built on this comparison. Reply with the title only.\n{compare}"],
    ["report",  "Write the report's opening paragraph, under 80 words.\nTitle: {title}\nComparison:\n{compare}"],
]

In [ ]:
NETWORK = {"drop_at": None}

def run_step(state, name, template):
    if name == NETWORK["drop_at"]:
        NETWORK["drop_at"] = None          # drops once, then works again
        raise ConnectionError(f"network dropped at step '{name}'")
    prompt = template.format(**PAPERS, **state.results)
    return chat([{"role": "user", "content": prompt}], max_tokens=250).content

## 2. No checkpoint: it breaks, so we start over

In [ ]:
def run_task(state, path=None):
    for name, template in state.steps:
        if name in state.results:
            continue                       # done before: skip it
        state.results[name] = run_step(state, name, template)
        print(f"  ✓ {name}")
        if path:
            save_state(state, path)      # checkpoint after every step
    return state

In [ ]:
USAGE["calls"] = 0
NETWORK["drop_at"] = "compare"
state = TaskState(task="2-page report on transformers for time series", steps=STEPS)
try:
    run_task(state)
except ConnectionError as e:
    print("💥", e)

state = TaskState(task="2-page report on transformers for time series", steps=STEPS)
run_task(state)
print(f"model calls: {USAGE['calls']}")

## 3. A checkpoint after every step

A checkpoint = a snapshot of state saved to a file (§18.6.2) · write to a temporary file first, then `os.replace` it over the real one in a single move:
if the program crashes **while writing**, the old checkpoint is still whole, not a half-written file

In [ ]:
def save_state(state, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(asdict(state), f, indent=1)
    os.replace(tmp, path)                  # one atomic move: never a half-written file

def load_state(path):
    with open(path) as f:
        data = json.load(f)
    if data.get("version") != STATE_VERSION:
        raise ValueError(f"checkpoint version {data.get('version')} ≠ {STATE_VERSION}: migrate it first")
    return TaskState(**data)

In [ ]:
USAGE["calls"] = 0
NETWORK["drop_at"] = "compare"
CKPT = "report_task.json"
state = TaskState(task="2-page report on transformers for time series", steps=STEPS)
try:
    run_task(state, CKPT)
except ConnectionError as e:
    print("💥", e)

In [ ]:
state = load_state(CKPT)
print("done before the crash:", list(state.results))
run_task(state, CKPT)
print(f"model calls: {USAGE['calls']}")

In [ ]:
saved = json.load(open(CKPT))
print("keys:", list(saved), "· version", saved["version"])
for name, out in saved["results"].items():
    print(f"  {name:8} {out[:70]!r}…")

In [ ]:
print(state.results["title"], "\n")
print(state.results["report"])

## 4. Rollback: undo the last k steps

Textbook §18.6.2: undo the latest actions when a mistake is found — here: the opening paragraph is too hard for first-year students.
Roll back the last two steps (title, report), change the plan, and redo only those two steps

In [ ]:
def rollback(state, k):
    for name in list(state.results)[-k:]:
        del state.results[name]
    return state

In [ ]:
USAGE["calls"] = 0
state = rollback(load_state(CKPT), 2)
state.steps[5][1] = "Write the report's opening paragraph for first-year students, under 60 words, no jargon.\nTitle: {title}\nComparison:\n{compare}"
run_task(state, CKPT)
print(f"model calls: {USAGE['calls']}\n\n{state.results['report']}")

In [ ]:
cost = USAGE["in"] / 1e6 * 0.09 + USAGE["out"] / 1e6 * 0.28
print(f"the whole lab used {USAGE['in']:,} input tokens + {USAGE['out']:,} output tokens ≈ ${cost:.4f}")

## Summary

| What we did | What we saw in the lab |
|---|---|
| State as a dataclass + version | Every field has a clear name · `load_state` refuses a file whose version doesn't match |
| No checkpoint | Network drop at step 4 → start over · **9** model calls for a 6-step task |
| A checkpoint after every step | Resume at step 4 · exactly **6** model calls |
| `os.replace` | The checkpoint file is never left half-written |
| Roll back 2 steps | A new opening paragraph for just **2** model calls |

**Exercises**
1. Add a `beliefs: dict` field (what the agent already knows, e.g. `{"audience": "first-year"}`) to `TaskState` and raise `STATE_VERSION` to 2 — can you still load the old `report_task.json`? Write a `migrate` function that turns a version-1 file into version 2
2. Make the network drop at the `report` step (the last one) instead — how many model calls does the checkpoint save?

**Next — Lab 3:** when everything fails — backoff + jitter, a fallback model, and catching an agent that loops